# Phases 2–3 — Agent development

## Phase 2: Risk Analyst Agent

This executable section exercises the five-step risk-analysis module through its public interface. Deterministic evidence extraction is intentionally separate from OpenAI classification so prompts, outputs, and audit events share the same facts. When an OpenAI client is configured, the same interface validates the model response as `RiskAnalystOutput`; this notebook uses the deterministic adapter so it runs without disclosing or requiring credentials.

In [1]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
if (cwd / "data").is_dir() and (cwd / "src").is_dir():
    STARTER = cwd
elif cwd.name == "notebooks":
    STARTER = cwd.parent
else:
    STARTER = cwd / "starter"
sys.path.insert(0, str(STARTER))

from src.foundation_sar import DataLoader, ExplainabilityLogger, RiskAnalystOutput
from src.risk_analyst_agent import RiskAnalystAgent

DATA = STARTER / "data"
AUDIT = STARTER / "outputs" / "audit_logs" / "phase2_notebook.jsonl"
AUDIT.parent.mkdir(parents=True, exist_ok=True)
if AUDIT.exists():
    AUDIT.unlink()

loader = DataLoader.load(DATA)
case = loader.get_case("CUST_0053")
logger = ExplainabilityLogger(AUDIT)
agent = RiskAnalystAgent(None, logger)
print({"case_id": case.case_id, "customer_id": case.customer.customer_id})

{'case_id': 'CASE-00866fe5-4080-455d-b137-c49ee339ac86', 'customer_id': 'CUST_0053'}


### Five-step reasoning contract

1. **Data Review** — customer, account, transaction, and screening facts.
2. **Pattern Recognition** — amount, timing, velocity, channel, counterparty, and geography.
3. **Regulatory Mapping** — relevant BSA/AML concepts without asserting a legal conclusion.
4. **Risk Quantification** — confidence from 0–1 and Low/Medium/High/Critical risk.
5. **Classification Decision** — exactly one of Structuring, Sanctions, Fraud, Money_Laundering, or Other.

Sanctions classification requires a confirmed result from an authoritative OFAC Sanctions List Service screening source; names, locations, or transaction text alone are insufficient.

In [2]:
evidence = agent.extract_case_evidence(case)
evidence

{'transaction_count': 94,
 'total_amount': 274025.18,
 'largest_transaction': 9915.61,
 'activity_start': '2025-02-06',
 'activity_end': '2025-07-31',
 'cash_deposits_9000_to_under_10000': 7,
 'cash_band_total': 67569.06,
 'wire_count': 10,
 'wire_total': 18061.24,
 'unique_counterparties': 23,
 'unique_locations': 9,
 'confirmed_ofac_matches': [],
 'missing_counterparty_count': 71,
 'missing_location_count': 70}

### Structured analysis

`CUST_0053` is used because its repeated cash deposits immediately below $10,000 provide a concentrated, auditable pattern. The output captures the pattern and confidence for downstream review.

In [3]:
result = agent.analyze_case(case)
result.model_dump(mode="json")

{'case_id': 'CASE-00866fe5-4080-455d-b137-c49ee339ac86',
 'suspicious_activity_type': 'Structuring',
 'confidence_score': 0.93,
 'risk_level': 'High',
 'reasoning': 'Data Review: 94 transactions totaling $274,025.18. Pattern Recognition: 7 cash deposits from $9,000 to under $10,000; Cash-band total $67,569.06. Regulatory Mapping: indicators are triage evidence and require human review. Risk Quantification: High at 0.93 confidence. Classification Decision: Structuring.',
 'suspicious_indicators': ['7 cash deposits from $9,000 to under $10,000',
  'Cash-band total $67,569.06']}

In [4]:
assert isinstance(result, RiskAnalystOutput)
assert result.suspicious_activity_type in {
    "Structuring", "Sanctions", "Fraud", "Money_Laundering", "Other"
}
assert 0 <= result.confidence_score <= 1
assert result.risk_level in {"Low", "Medium", "High", "Critical"}
assert logger.entries[-1]["success"] is True
print({
    "classification": result.suspicious_activity_type,
    "confidence": result.confidence_score,
    "risk_level": result.risk_level,
    "audit_events": len(logger.entries),
})

{'classification': 'Structuring', 'confidence': 0.93, 'risk_level': 'High', 'audit_events': 1}


### Robust model-output parsing

The OpenAI adapter accepts a plain JSON object or an object inside explanatory text/code fences, then validates every field through Pydantic. Empty, malformed, out-of-range, or unsupported outputs fail closed and are audited.

In [5]:
sample = '''Analysis follows:
```json
{"classification":"Fraud","confidence_score":0.82,"reasoning":"Observed unauthorized activity","key_indicators":["unauthorized transfer"],"risk_level":"High"}
```'''
parsed_text = agent._extract_json_from_response(sample)
print(parsed_text)

try:
    agent._extract_json_from_response("not structured output")
except ValueError as exc:
    print(f"Expected parser failure: {exc}")

{"classification":"Fraud","confidence_score":0.82,"reasoning":"Observed unauthorized activity","key_indicators":["unauthorized transfer"],"risk_level":"High"}
Expected parser failure: No JSON content found


## Phase 2 result

- The prompt names all five required reasoning steps and classifications.
- Evidence extraction is deterministic and excludes transaction-ID hints.
- OpenAI output is parsed and validated as a structured Pydantic model.
- API and parsing failures are recorded as unsuccessful audit events.
- An offline deterministic adapter keeps the workflow executable; configured deployments use the injected OpenAI client through the same interface.

Phase 3 extends this notebook with the compliance narrative agent.

# Phase 3 — Compliance Officer Agent

The Compliance Officer runs only after a human approves the risk analysis. Its ReACT implementation verifies facts, selects material five-element evidence, drafts a concise narrative, and concludes with a source-controlled citation set. Research grounding is recorded in the repository's `regulatory_research.md` from official FinCEN, eCFR, FFIEC, OFAC, and U.S. Code sources.

The course's 120-word cap applies to generated narratives.

In [6]:
from src.compliance_officer_agent import REGULATORY_SOURCES, ComplianceOfficerAgent

COMPLIANCE_AUDIT = STARTER / "outputs" / "audit_logs" / "phase3_notebook.jsonl"
if COMPLIANCE_AUDIT.exists():
    COMPLIANCE_AUDIT.unlink()

compliance_logger = ExplainabilityLogger(COMPLIANCE_AUDIT)
compliance_agent = ComplianceOfficerAgent(None, compliance_logger)
risk_result = result
compliance_result = compliance_agent.generate_compliance_narrative(case, risk_result)
compliance_result.model_dump(mode="json")

{'case_id': 'CASE-00866fe5-4080-455d-b137-c49ee339ac86',
 'sar_narrative': 'Who: Tracy Lewis (CUST_0053), using CUST_0053_ACC_1, CUST_0053_ACC_2, CUST_0053_ACC_3. What: conducted 94 transactions totaling $274,025.18. When: activity occurred from 2025-02-06 through 2025-07-31. Where: activity involved Branch_Airport_Terminal, Branch_Downtown_Main, Branch_Eastside_Center. Why: observed facts—7 cash deposits from $9,000 to under $10,000; Cash-band total $67,569.06—may indicate structuring and require human review. Supporting transaction records are retained; unknown counterparties or locations remain unverified.',
 'regulatory_citations': ['31 C.F.R. § 1010.311 (currency transaction reports)',
  '31 C.F.R. § 1010.314 (aggregation)',
  '31 C.F.R. § 1020.320(a)(2)(ii) (transactions designed to evade BSA requirements)',
  'FinCEN, Guidance on Preparing a Complete and Sufficient SAR Narrative'],
 'completeness_check': True,
 'reasoning': 'ReACT result: verified case and risk facts; selected t

## ReACT and factual completeness

- **Reasoning:** verify case/risk facts, identify unknowns, and select only applicable primary sources.
- **Action:** assemble who, what, when, where, and why in chronological, attributable language.
- **Conclusion:** return one Pydantic-validated narrative no longer than 120 words with controlled citations.

The deterministic validator checks every five-element signal, exact word count, and nonempty citations.

In [7]:
checklist = compliance_agent.validate_narrative(case, risk_result, compliance_result)
print(checklist)
print({
    "word_count": compliance_result.word_count,
    "citation_count": len(compliance_result.regulatory_citations),
    "complete": compliance_result.completeness_check,
})
assert compliance_result.word_count <= 120
assert compliance_result.word_count == len(compliance_result.sar_narrative.split())
assert checklist["complete"] is True
assert compliance_result.regulatory_citations == REGULATORY_SOURCES[risk_result.suspicious_activity_type]
assert compliance_logger.entries[-1]["success"] is True

{'who': True, 'what': True, 'when': True, 'where': True, 'why': True, 'within_word_limit': True, 'has_citations': True, 'complete': True}
{'word_count': 57, 'citation_count': 4, 'complete': True}


## Regulatory guardrails demonstrated

- A bank SAR trigger and the $10,000 CTR threshold are distinct concepts.
- Repeated transactions below the CTR threshold can be review evidence but are not proof of intent.
- The narrative uses *may indicate* and *requires human review* rather than declaring a crime.
- Sanctions narratives require a confirmed authoritative OFAC Sanctions List Service screening result.
- Missing counterparties or locations are disclosed as unknown rather than invented.
- Only the source-controlled citation set for the selected classification is emitted.

In [8]:
print(compliance_result.sar_narrative)
print()
print("Citations:")
for citation in compliance_result.regulatory_citations:
    print(f"- {citation}")

Who: Tracy Lewis (CUST_0053), using CUST_0053_ACC_1, CUST_0053_ACC_2, CUST_0053_ACC_3. What: conducted 94 transactions totaling $274,025.18. When: activity occurred from 2025-02-06 through 2025-07-31. Where: activity involved Branch_Airport_Terminal, Branch_Downtown_Main, Branch_Eastside_Center. Why: observed facts—7 cash deposits from $9,000 to under $10,000; Cash-band total $67,569.06—may indicate structuring and require human review. Supporting transaction records are retained; unknown counterparties or locations remain unverified.

Citations:
- 31 C.F.R. § 1010.311 (currency transaction reports)
- 31 C.F.R. § 1010.314 (aggregation)
- 31 C.F.R. § 1020.320(a)(2)(ii) (transactions designed to evade BSA requirements)
- FinCEN, Guidance on Preparing a Complete and Sufficient SAR Narrative


## Phases 2–3 result

Both agent modules expose one deep interface each: `analyze_case` and `generate_compliance_narrative`. OpenAI clients are injected at those seams; the offline adapters exercise the same validation and logging contracts. Model responses cannot bypass Pydantic classification, confidence, risk-level, word-count, or citation controls. Every success and failure appends a structured audit event.